In [106]:
#Import all the necessary modules
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
import seaborn as sns
import random
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
# calculate accuracy measures and confusion matrix
from sklearn import metrics
num_bins = 10


In [147]:
dbody=pd.read_csv("data/diabetes.csv")
dmind=pd.read_csv("data/Mental_Health_Dataset.csv")
print(f"Body Columns are:{list(dbody.columns)}")
print(f"Mind Columns are:{list(dmind.columns)}")




Body Columns are:['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
Mind Columns are:['Timestamp', 'Gender', 'Country', 'Occupation', 'self_employed', 'family_history', 'treatment', 'Days_Indoors', 'Growing_Stress', 'Changes_Habits', 'Mental_Health_History', 'Mood_Swings', 'Coping_Struggles', 'Work_Interest', 'Social_Weakness', 'mental_health_interview', 'care_options']


In [108]:
print(dbody)

     Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0              6      148             72             35        0  33.6   
1              1       85             66             29        0  26.6   
2              8      183             64              0        0  23.3   
3              1       89             66             23       94  28.1   
4              0      137             40             35      168  43.1   
..           ...      ...            ...            ...      ...   ...   
763           10      101             76             48      180  32.9   
764            2      122             70             27        0  36.8   
765            5      121             72             23      112  26.2   
766            1      126             60              0        0  30.1   
767            1       93             70             31        0  30.4   

     DiabetesPedigreeFunction  Age  Outcome  
0                       0.627   50        1  
1                  

In [149]:
print(dmind)

              Timestamp  Gender        Country Occupation self_employed  \
0       8/27/2014 11:29  Female  United States  Corporate           NaN   
1       8/27/2014 11:31  Female  United States  Corporate           NaN   
2       8/27/2014 11:32  Female  United States  Corporate           NaN   
3       8/27/2014 11:37  Female  United States  Corporate            No   
4       8/27/2014 11:43  Female  United States  Corporate            No   
...                 ...     ...            ...        ...           ...   
292359  7/27/2015 23:25    Male  United States   Business           Yes   
292360   8/17/2015 9:38    Male   South Africa   Business            No   
292361  8/25/2015 19:59    Male  United States   Business            No   
292362   9/26/2015 1:07    Male  United States   Business            No   
292363   2/1/2016 23:04    Male  United States   Business            No   

       family_history treatment Days_Indoors Growing_Stress Changes_Habits  \
0                  No

In [150]:
dbody.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [151]:
print(dmind.dtypes)

Timestamp                  object
Gender                     object
Country                    object
Occupation                 object
self_employed              object
family_history             object
treatment                  object
Days_Indoors               object
Growing_Stress             object
Changes_Habits             object
Mental_Health_History      object
Mood_Swings                object
Coping_Struggles           object
Work_Interest              object
Social_Weakness            object
mental_health_interview    object
care_options               object
dtype: object


In [152]:
dmind.drop(['Country','Timestamp'], axis=1, inplace=True)

In [153]:
summary = {}

for col in dmind.columns:
    summary[col] = {
        "unique_count": dmind[col].nunique(),
        "top_values": dmind[col].value_counts(dropna=False).head(5).to_dict()
    }

summary

{'Gender': {'unique_count': 2,
  'top_values': {'Male': 239850, 'Female': 52514}},
 'Occupation': {'unique_count': 5,
  'top_values': {'Housewife': 66351,
   'Student': 61794,
   'Corporate': 61229,
   'Others': 52841,
   'Business': 50149}},
 'self_employed': {'unique_count': 2,
  'top_values': {'No': 257994, 'Yes': 29168, nan: 5202}},
 'family_history': {'unique_count': 2,
  'top_values': {'No': 176832, 'Yes': 115532}},
 'treatment': {'unique_count': 2, 'top_values': {'Yes': 147606, 'No': 144758}},
 'Days_Indoors': {'unique_count': 5,
  'top_values': {'1-14 days': 63548,
   '31-60 days': 60705,
   'Go out Every day': 58366,
   'More than 2 months': 55916,
   '15-30 days': 53829}},
 'Growing_Stress': {'unique_count': 3,
  'top_values': {'Maybe': 99985, 'Yes': 99653, 'No': 92726}},
 'Changes_Habits': {'unique_count': 3,
  'top_values': {'Yes': 109523, 'Maybe': 95166, 'No': 87675}},
 'Mental_Health_History': {'unique_count': 3,
  'top_values': {'No': 104018, 'Maybe': 95378, 'Yes': 92968

In [154]:
for col in dmind.columns:
    print("\n", col)
    print(dmind[col].unique())


 Gender
['Female' 'Male']

 Occupation
['Corporate' 'Student' 'Business' 'Housewife' 'Others']

 self_employed
[nan 'No' 'Yes']

 family_history
['No' 'Yes']

 treatment
['Yes' 'No']

 Days_Indoors
['1-14 days' 'Go out Every day' 'More than 2 months' '15-30 days'
 '31-60 days']

 Growing_Stress
['Yes' 'No' 'Maybe']

 Changes_Habits
['No' 'Yes' 'Maybe']

 Mental_Health_History
['Yes' 'No' 'Maybe']

 Mood_Swings
['Medium' 'Low' 'High']

 Coping_Struggles
['No' 'Yes']

 Work_Interest
['No' 'Maybe' 'Yes']

 Social_Weakness
['Yes' 'No' 'Maybe']

 mental_health_interview
['No' 'Maybe' 'Yes']

 care_options
['Not sure' 'No' 'Yes']


In [117]:
male_terms = ['male', 'm', 'man', 'cis male', 'male-ish', 'maile', 'msle', 'mal', 'make', 'mail', 'malr']

female_terms = ['female', 'f', 'woman', 'cis female', 'femake', 'femail']

non_binary_terms = ['non-binary', 'queer', 'genderqueer', 'androgyne', 'agender', 'neuter', 'fluid']
def clean_gender(x):
    if x in male_terms:
        return 0
    elif x in female_terms:
        return 1
    else:
        return 2
dmind['Gender'] = dmind['Gender'].apply(clean_gender)

In [118]:
dmind = dmind.applymap(lambda x: str(x).strip().lower() if isinstance(x, str) else x)

In [119]:
def simple_map(x):
    if x == "yes":
        return 1
    elif x == "no":
        return 0
    else:
        return 0.5

In [120]:
yes_no_cols = [
    'self_employed',
    'family_history',
    'remote_work',
    'tech_company',
    'benefits',
    'care_options',
    'wellness_program',
    'seek_help',
    'anonymity',
    'mental_health_consequence',
    'phys_health_consequence',
    'mental_health_interview',
    'phys_health_interview',
    'obs_consequence',
    'treatment',
    'supervisor',
    'coworkers',
    'mental_vs_physical'
]
for col in yes_no_cols:
    dmind[col] = dmind[col].apply(simple_map)

In [121]:
work_map = {
    'never': 0,
    'rarely': 1,
    'sometimes': 2,
    'often': 3
}

dmind['work_interfere'] = dmind['work_interfere'].map(work_map)

In [122]:
emp_map = {
    '1-5': 0,
    '6-25': 1,
    '26-100': 2,
    '100-500': 3,
    '500-1000': 4,
    'more than 1000': 5
}

dmind['no_employees'] = dmind['no_employees'].map(emp_map)

In [123]:
leave_map = {
    'very difficult': 0,
    'somewhat difficult': 1,
    "don't know": 2,
    'somewhat easy': 3,
    'very easy': 4
}

dmind['leave'] = dmind['leave'].map(leave_map)

In [124]:
dmind['work_interfere_missing'] = dmind['work_interfere'].isna().astype(int)
dmind['work_interfere'] = dmind['work_interfere'].fillna(2)

In [125]:
dmind

,Age,Gender,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,...,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,work_interfere_missing
0,37,2,0.5,0,1,3.0,1,0,1,1.0,...,3,0.0,0.0,0.5,1.0,0.0,0.5,1.0,0,0
1,44,2,0.5,0,0,1.0,5,0,0,0.5,...,2,0.5,0.0,0.0,0.0,0.0,0.0,0.5,0,0
2,32,2,0.5,0,0,1.0,1,0,1,0.0,...,1,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0,0
3,31,2,0.5,1,1,3.0,2,0,1,0.0,...,1,1.0,1.0,0.5,0.0,0.5,0.5,0.0,1,0
4,31,2,0.5,0,0,0.0,3,1,1,1.0,...,2,0.0,0.0,0.5,1.0,1.0,1.0,0.5,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1254,26,0,0.0,0,1,2.0,2,0,1,0.0,...,3,0.0,0.0,0.5,0.5,0.0,0.0,0.5,0,1
1255,32,2,0.0,1,1,3.0,2,1,1,1.0,...,1,0.0,0.0,0.5,1.0,0.0,0.0,1.0,0,0
1256,34,0,0.0,1,1,2.0,5,0,1,1.0,...,1,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0,0
1257,46,1,0.0,0,0,2.0,3,1,1,0.0,...,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0,1


In [126]:
dmind.isnull().sum()

Age                          0
Gender                       0
self_employed                0
family_history               0
treatment                    0
work_interfere               0
no_employees                 0
remote_work                  0
tech_company                 0
benefits                     0
care_options                 0
wellness_program             0
seek_help                    0
anonymity                    0
leave                        0
mental_health_consequence    0
phys_health_consequence      0
coworkers                    0
supervisor                   0
mental_health_interview      0
phys_health_interview        0
mental_vs_physical           0
obs_consequence              0
work_interfere_missing       0
dtype: int64

In [127]:
dmind.dtypes


Age                            int64
Gender                         int64
self_employed                float64
family_history                 int64
treatment                      int64
work_interfere               float64
no_employees                   int64
remote_work                    int64
tech_company                   int64
benefits                     float64
care_options                 float64
wellness_program             float64
seek_help                    float64
anonymity                    float64
leave                          int64
mental_health_consequence    float64
phys_health_consequence      float64
coworkers                    float64
supervisor                   float64
mental_health_interview      float64
phys_health_interview        float64
mental_vs_physical           float64
obs_consequence                int64
work_interfere_missing         int64
dtype: object

In [128]:
dmind['treatment'] = dmind['treatment'].astype(int)

In [129]:
dmind = dmind.fillna(0)
dbody=dbody.fillna(0)

In [130]:
Xbody = dbody.drop('Outcome', axis=1) # Copying all the predictor variables into X dataframe. 'Final_grade' is dropped as it is dependent variable
Ybody = dbody['Outcome']# Copy the 'Final_grade' column alone into the y dataframe. This is the dependent variable 
seed=25
#now we break the X and y dataframes into training set and test set. For this we will use
#Sklearn package's data splitting function which is based on random function.
Xbody_train, Xbody_test, Ybody_train, Ybody_test = train_test_split(Xbody, Ybody, test_size=0.2, random_state=25,stratify=Ybody)# Splitting X and y into training and test set in 80:20 ratio

In [131]:
# Import the Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest model with a fixed random state for reproducibility
rf_model_body = RandomForestClassifier(class_weight='balanced',random_state=25)

# Train the model on the training data
rf_model_body.fit(Xbody_train, Ybody_train)

# Predict the target values for the test data
rf_pred_body = rf_model_body.predict(Xbody_test)


In [132]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(Ybody_test, rf_pred_body))# Calculating the accuracy score of the model
print(classification_report(Ybody_test, rf_pred_body))
print("CONFUSION MATRIX:",metrics.confusion_matrix(Ybody_test, rf_pred_body))#Printing the confusion matrix to evaluate prediction results

Accuracy: 0.7337662337662337
              precision    recall  f1-score   support

           0       0.76      0.86      0.81       100
           1       0.66      0.50      0.57        54

    accuracy                           0.73       154
   macro avg       0.71      0.68      0.69       154
weighted avg       0.73      0.73      0.72       154

CONFUSION MATRIX: [[86 14]
 [27 27]]


In [133]:
Xmind = dmind.drop('treatment', axis=1) # Copying all the predictor variables into X dataframe. 'Final_grade' is dropped as it is dependent variable
Ymind = dmind['treatment']# Copy the 'Final_grade' column alone into the y dataframe. This is the dependent variable 
seed=25
#now we break the X and y dataframes into training set and test set. For this we will use
#Sklearn package's data splitting function which is based on random function.
Xmind_train, Xmind_test, Ymind_train, Ymind_test = train_test_split(Xmind, Ymind, test_size=0.2, random_state=25,stratify=Ymind)# Splitting X and y into training and test set in 80:20 ratio

In [134]:
# Import the Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest model with a fixed random state for reproducibility
rf_model_mental = RandomForestClassifier(class_weight='balanced',random_state=25)

# Train the model on the training data
rf_model_mental.fit(Xmind_train, Ymind_train)

# Predict the target values for the test data
rf_pred_mental = rf_model_mental.predict(Xmind_test)

In [135]:
from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(Ymind_test, rf_pred_mental))# Calculating the accuracy score of the model
print(classification_report(Ymind_test, rf_pred_mental))
print("CONFUSION MATRIX:",metrics.confusion_matrix(Ymind_test, rf_pred_mental))#Printing the confusion matrix to evaluate prediction results

Accuracy: 0.8605577689243028
              precision    recall  f1-score   support

           0       0.89      0.82      0.85       124
           1       0.84      0.90      0.87       127

    accuracy                           0.86       251
   macro avg       0.86      0.86      0.86       251
weighted avg       0.86      0.86      0.86       251

CONFUSION MATRIX: [[102  22]
 [ 13 114]]


In [136]:
import joblib
import os

os.makedirs("models", exist_ok=True)

# Save physical model
joblib.dump(rf_model_body,   "models/physical_model.pkl")

# Save mental model
joblib.dump(rf_model_mental, "models/mental_model.pkl")

print("✅ Both models saved successfully!")

✅ Both models saved successfully!


In [137]:
from recommend import get_combined_health_index

score, status, color, emoji = get_combined_health_index(0.65, 0.72)
print(f"{emoji} Combined Risk: {score}% — {status}")

🟡 Combined Risk: 68.5% — Needs Some Attention


In [138]:
print(rf_model_mental.feature_names_in_)

['Age' 'Gender' 'self_employed' 'family_history' 'work_interfere'
 'no_employees' 'remote_work' 'tech_company' 'benefits' 'care_options'
 'wellness_program' 'seek_help' 'anonymity' 'leave'
 'mental_health_consequence' 'phys_health_consequence' 'coworkers'
 'supervisor' 'mental_health_interview' 'phys_health_interview'
 'mental_vs_physical' 'obs_consequence' 'work_interfere_missing']
